# InfoNCE 正负样本分布（数据侧）

只读 manifest，不加载模型。回答三个问题：

1. **target 重复度**：同一 batch 里有多少负样本的 target 文本和正样本完全相同（假负样本）。
2. **现有 `group_id` 屏蔽覆盖了多少**：`bidirectional_infonce` 只屏蔽同 `group_id` 的对；manifest 没有 `group_id` 字段时它不生效。
3. **同场景负样本占比**：同一房间的样本互为负样本的比例。

`B` 取"全局 batch"：跨卡 gather 后是 `batch_size × GPU 数`（VGGT 配置 64×2=128），单卡是 64。

In [ ]:
import json, os, random, re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

ROOT = Path(os.environ.get("VJEPA_ROOT", "/glob/g01-cache/pf/Yushuo/vjepa201"))
MANIFEST = ROOT / "data/spar_sftqa_sftvljepa_uni8.jsonl"   # 换成要查的训练 manifest
N_SAMPLE = 300_000        # 蓄水池抽样行数；None = 全读
BATCH_SIZES = [64, 128, 256]
N_TRIALS = 2000           # 每个 B 模拟多少个随机 batch
SEED = 0

In [ ]:
def scene_of(path: str):
    """.../<source>/images/<scene>/... -> (source, scene)；不匹配时退回父目录。"""
    parts = Path(path).parts
    if "images" in parts:
        i = parts.index("images")
        if 0 < i < len(parts) - 1:
            return parts[i - 1], parts[i + 1]
    return "?", str(Path(path).parent)

def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def load_rows(path, n_sample, seed):
    rng = random.Random(seed)
    keep, seen = [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            seen += 1
            if n_sample is None or len(keep) < n_sample:
                keep.append(line)
            else:
                j = rng.randrange(seen)
                if j < n_sample:
                    keep[j] = line
    rows = []
    for line in keep:
        r = json.loads(line)
        imgs = r.get("images") or [r.get("image")]
        source, scene = scene_of(imgs[0])
        rows.append({
            "query": r.get("query", ""),
            "target": r["target"],
            "target_norm": norm_text(r["target"]),
            "group_id": r.get("group_id"),
            "type": r.get("type"),
            "qa_kind": r.get("qa_kind"),
            "source": source,
            "scene": f"{source}/{scene}",
            "n_images": len(imgs),
            "n_unique_images": len(set(imgs)),
        })
    return rows, seen

rows, total = load_rows(MANIFEST, N_SAMPLE, SEED)
print(f"{MANIFEST.name}: 文件共 {total:,} 行，抽样 {len(rows):,} 行")
print("原始字段:", sorted(json.loads(open(MANIFEST, encoding='utf-8').readline()).keys()))

## 1. 概览：`group_id` / `type` 是否存在，帧是否重复

In [ ]:
N = len(rows)
def pct(k): return f"{100 * k / N:6.2f}%"

print("有 group_id 的行:", pct(sum(r["group_id"] is not None for r in rows)),
      " <- 0% 说明现有 group 屏蔽对这个 manifest 完全不生效")
print("有 type 的行    :", pct(sum(r["type"] is not None for r in rows)),
      " <- 0% 说明转换时没加 --include-metadata，下面按 target 形态分组")
print("source 分布     :", dict(Counter(r["source"] for r in rows).most_common()))
print("场景数          :", len({r["scene"] for r in rows}))
print()
print("每行图像数      :", dict(sorted(Counter(r["n_images"] for r in rows).items())))
print("每行不同图像数  :", dict(sorted(Counter(r["n_unique_images"] for r in rows).items())),
      " <- 小于图像数 = 循环补帧（单视图/少视图题被补到 8 帧）")

## 2. target 文本重复度

`Σp²` = 随机两行 target 相同的概率，也就是 batch 里任意一个负样本是假负样本的概率。

In [ ]:
NUM_RE = re.compile(r"^[-+]?\d+(?:\.\d+)?(?:\s*(?:m|cm|mm|meters?|metres?|centimeters?|degrees?|°|%))?\.?$")

def target_kind(t: str) -> str:
    if NUM_RE.match(t):
        return "numeric"
    n = len(t.split())
    if n <= 3:
        return "short(<=3w)"
    if re.search(r"\d", t):
        return "sentence+number"
    return "sentence"

for r in rows:
    r["kind"] = target_kind(r["target_norm"])

def dup_stats(sub):
    c = Counter(r["target_norm"] for r in sub)
    n = len(sub)
    p = np.array(list(c.values()), dtype=np.float64) / n
    return {"n": n, "unique": len(c), "sum_p2": float((p ** 2).sum()), "top1": c.most_common(1)[0]}

s = dup_stats(rows)
print(f"全体: n={s['n']:,}  不同 target={s['unique']:,} ({100*s['unique']/s['n']:.1f}%)  "
      f"Σp²={s['sum_p2']:.4f}  最高频={s['top1'][0]!r} ({100*s['top1'][1]/s['n']:.2f}%)")
print()
print("最高频 30 个 target:")
for t, k in Counter(r["target_norm"] for r in rows).most_common(30):
    print(f"  {100*k/N:6.2f}%  {k:>8,}  {t[:100]!r}")

In [ ]:
def table(key):
    groups = defaultdict(list)
    for r in rows:
        groups[r[key] if r[key] is not None else "<none>"].append(r)
    print(f"{key:34s} {'n':>9s} {'share':>7s} {'unique%':>8s} {'Σp²':>8s} {'top1%':>7s}  top1 / 词数中位")
    for g, sub in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        s = dup_stats(sub)
        med = int(np.median([len(r["target_norm"].split()) for r in sub]))
        print(f"{str(g)[:34]:34s} {s['n']:>9,} {100*s['n']/N:6.1f}% {100*s['unique']/s['n']:7.1f}% "
              f"{s['sum_p2']:8.4f} {100*s['top1'][1]/s['n']:6.1f}%  {s['top1'][0][:40]!r} / {med}")

table("kind")
print()
table("type" if any(r["type"] for r in rows) else "source")
print()
print("top1% 也是该组的纯文本盲猜上限：只看题型不看图就能答对的比例。")

## 3. 模拟 batch：正样本、假负样本、同场景负样本

每个 anchor 有 1 个正样本和 `B-1` 个负样本。统计负样本里：

- **同 target**：文本完全相同，是假负样本。
- **同场景**：同一房间，图像内容高度相关。
- **已被 group 屏蔽**：同 `group_id`，现有代码已从 softmax 里去掉。
- **屏蔽后仍剩的假负**：同 target 但不同 `group_id`。

In [ ]:
def codes(values, none_unique=False):
    table_, out = {}, np.empty(len(values), dtype=np.int64)
    for i, v in enumerate(values):
        out[i] = -(i + 1) if (v is None and none_unique) else table_.setdefault(v, len(table_))
    return out

t_code = codes([r["target_norm"] for r in rows])
s_code = codes([r["scene"] for r in rows])
g_code = codes([r["group_id"] for r in rows], none_unique=True)
tg_code = codes(list(zip(t_code.tolist(), g_code.tolist())))

def pair_counts(c):
    """每个 anchor 在 batch 内与它同码的其他样本数。"""
    _, inv, cnt = np.unique(c, return_inverse=True, return_counts=True)
    return cnt[inv] - 1

rng = np.random.default_rng(SEED)
print(f"{'B':>4s} | {'同target负样本':>14s} {'有假负的anchor':>14s} {'每anchor假负数':>14s} | "
      f"{'同场景负样本':>12s} {'有同场景的anchor':>16s} | {'已被group屏蔽':>13s} {'屏蔽后仍剩假负':>14s}")
sim = {}
for B in BATCH_SIZES:
    acc = defaultdict(list)
    for _ in range(N_TRIALS):
        idx = rng.choice(N, size=B, replace=False)
        same_t = pair_counts(t_code[idx]); same_s = pair_counts(s_code[idx])
        same_g = pair_counts(g_code[idx]); same_tg = pair_counts(tg_code[idx])
        acc["t_pair"].append(same_t.sum() / (B * (B - 1)))
        acc["t_anchor"].append((same_t > 0).mean())
        acc["t_per"].append(same_t.mean())
        acc["s_pair"].append(same_s.sum() / (B * (B - 1)))
        acc["s_anchor"].append((same_s > 0).mean())
        acc["g_pair"].append(same_g.sum() / (B * (B - 1)))
        acc["left_pair"].append((same_t - same_tg).sum() / (B * (B - 1)))
    m = {k: float(np.mean(v)) for k, v in acc.items()}
    sim[B] = m
    print(f"{B:>4d} | {100*m['t_pair']:13.2f}% {100*m['t_anchor']:13.1f}% {m['t_per']:14.2f} | "
          f"{100*m['s_pair']:11.3f}% {100*m['s_anchor']:15.1f}% | {100*m['g_pair']:12.3f}% {100*m['left_pair']:13.2f}%")

## 4. 按题型分开看

训练 batch 是全体混合抽样，所以某个题型的 anchor 遇到假负的概率 ≈ 它自己的 target 在全体里的占比 × (B-1)。
下面按 anchor 所属组统计，找出哪类题贡献了大部分假负样本。

In [ ]:
B = BATCH_SIZES[min(1, len(BATCH_SIZES) - 1)]
key = "type" if any(r["type"] for r in rows) else "kind"
labels = np.array([str(r[key]) for r in rows])
hit = defaultdict(list); per = defaultdict(list)
rng = np.random.default_rng(SEED + 1)
for _ in range(N_TRIALS):
    idx = rng.choice(N, size=B, replace=False)
    same_t = pair_counts(t_code[idx])
    for lab, v in zip(labels[idx], same_t):
        hit[lab].append(v > 0); per[lab].append(v)

total_fn = sum(sum(v) for v in per.values())
print(f"B={B}，按 {key} 分组")
print(f"{key:34s} {'anchor占比':>10s} {'有假负的anchor':>14s} {'每anchor假负数':>14s} {'占全部假负':>10s}")
n_all = sum(len(v) for v in hit.values())
for lab in sorted(hit, key=lambda k: -sum(per[k])):
    print(f"{lab[:34]:34s} {100*len(hit[lab])/n_all:9.1f}% {100*np.mean(hit[lab]):13.1f}% "
          f"{np.mean(per[lab]):14.2f} {100*sum(per[lab])/max(total_fn,1):9.1f}%")

## 5. 实例：一个随机 batch 里的重复组，以及高频 target 对应的 query

In [ ]:
B = BATCH_SIZES[0]
idx = np.random.default_rng(SEED + 2).choice(N, size=B, replace=False)
groups = defaultdict(list)
for i in idx:
    groups[rows[i]["target_norm"]].append(rows[i])
dups = {t: rs for t, rs in groups.items() if len(rs) > 1}
print(f"随机 batch (B={B}): {len(groups)} 个不同 target，{len(dups)} 个重复组，"
      f"{sum(len(v) for v in dups.values())} 个样本带假负样本\n")
for t, rs in sorted(dups.items(), key=lambda kv: -len(kv[1])):
    print(f"target={t[:80]!r}  x{len(rs)}")
    for r in rs[:4]:
        print(f"    [{r['type'] or r['kind']}] {r['scene']}  Q: {r['query'][:110]!r}")

In [ ]:
by_target = defaultdict(list)
for r in rows:
    by_target[r["target_norm"]].append(r)
rng = random.Random(SEED)
for t, k in Counter(r["target_norm"] for r in rows).most_common(8):
    print(f"\n=== target={t[:80]!r}  ({100*k/N:.2f}%) ===")
    for r in rng.sample(by_target[t], k=min(3, k)):
        print(f"  [{r['type'] or r['kind']}] Q: {r['query'][:160]!r}")

## 怎么读

- 第 3 节"有假负的 anchor"高（例如 >30%）：InfoNCE 被迫把相同的 target 向量推开，loss 有下限，梯度有噪声。
- "已被 group 屏蔽"≈0 而"屏蔽后仍剩假负"≈"同 target 负样本"：现有 `group_id` 机制没覆盖这个问题。
- 第 4 节若少数题型（数字、方向词）贡献了大部分假负：在数据侧降采样或改写成句子，比改 loss 更直接。
- 第 2 节 `top1%` 高的题型：不看图也能猜对，评测时要和这个盲猜基线比。